In [ ]:
##################################################################
##          This project is aimed to analyze the share prices of
##          dhaka stock exchange and make necessary charts and
##             investment decisions.
##      Th bdshare package provides necessary classes for forking
##      and importing current and last two years market prices of
##      all instruments.

# coded by Md. Anisur Rahman Bali
#################################################################

In [ ]:
!pip install bdshare

In [ ]:
# get list of all share codes
import os
import pandas as pd
from bdshare import get_current_trading_code, get_last_trade_price_data
from bdshare import get_historical_data
from datetime import datetime

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive = 'drive/MyDrive/Stock Market Analysis/'

trading_codes = get_current_trading_code()


ERROR:bdshare.util.helper:Request error on https://dsebd.org/latest_share_price_scroll_l.php (attempt 1/3): HTTPSConnectionPool(host='dsebd.org', port=443): Max retries exceeded with url: /latest_share_price_scroll_l.php (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
ERROR:bdshare.util.helper:Request error on https://dsebd.com.bd/latest_share_price_scroll_l.php (attempt 1/3): HTTPSConnectionPool(host='dsebd.com.bd', port=443): Max retries exceeded with url: /latest_share_price_scroll_l.php (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7d8942dbd8e0>: Failed to resolve 'dsebd.com.bd' ([Errno -2] Name or service not known)"))
ERROR:bdshare.util.helper:Request error on https://dsebd.org/latest_share_price_scroll_l.php (attempt 2/3): HTTPSConnectionPool(host='dsebd.org', port=443): Max retries exceeded with url: /latest_share_price_scroll_l

BDShareError: Failed to fetch data after 3 retries. URLs tried: ['https://dsebd.org/latest_share_price_scroll_l.php', 'https://dsebd.com.bd/latest_share_price_scroll_l.php']. Last error: HTTPSConnectionPool(host='dsebd.com.bd', port=443): Max retries exceeded with url: /latest_share_price_scroll_l.php (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7d8942dbe2d0>: Failed to resolve 'dsebd.com.bd' ([Errno -2] Name or service not known)"))

In [ ]:

# get latest trading price data from scroll bar
cp =  get_last_trade_price_data()
cp = cp.rename(columns={cp.columns[0]: "code_price"})

## make a dataframe of trading price and code
tp_df = pd.DataFrame(columns=['code', 'price'])

for i in cp['code_price']:
    tc = i.split()[0]   #trading code
    price = i.split()[-1]
    new_line = pd.DataFrame([[tc, price]], columns=['code', 'price'])
    # tp_df = tp_df.append({'code': tc, 'price': price}, ignore_index=True)


    tp_df = pd.concat([tp_df, new_line])

ERROR:bdshare.stock.trading:Attempt 1 failed for quotes.txt: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)>
ERROR:bdshare.stock.trading:Attempt 2 failed for quotes.txt: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)>
ERROR:bdshare.stock.trading:Attempt 3 failed for quotes.txt: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)>


BDShareError: Failed to fetch quotes.txt after 3 retries.

In [ ]:
## exporting the dataframe
today = datetime.today().strftime('%Y-%m-%d')
db_existing = pd.read_csv(drive+"historical_price_data.csv")

last_date_record = db_existing["date"].max()
print(last_date_record)

df_recent = get_historical_data(last_date_record,today)

df_recent = df_recent.reset_index()

In [ ]:
db_merged = pd.concat([db_existing, df_recent], ignore_index=True)

db = db_merged.drop_duplicates(subset=["date", "symbol"], ignore_index=True)
db = db.sort_values(by=['symbol', 'date'], ascending = [True, False])
db.to_csv(drive+"historical_price_data_merged.csv")

In [ ]:
## check if today's price is below criteria

# add stock in the portfolio
for s in trading_codes['symbol']:

    df = db[db['symbol']==s]

    df['ltp'] = pd.to_numeric(df['ltp'])

    # remove the values with 0
    df = df[df['ltp']!=0]
    date = today

    last_days_ltp = df['ltp'][5:350]

    last_days_sd = last_days_ltp.std()
    last_days_avg = last_days_ltp.mean()

    ltp_now = tp_df[tp_df.code==s]['price'].values

    if len(ltp_now)>0:
        ltp_now = float(ltp_now[0])

        if (ltp_now <= last_days_avg - 2*last_days_sd) and last_days_sd!=0 and ltp_now!=0:

                print(f"Buy the stock {s} at Taka {ltp_now} on {date} \n https://www.dsebd.org/displayCompany.php?name={s} \n graph https://www.dsebd.org/php_graph/monthly_graph.php?inst={s}&duration=3&type=price \n \n")